In [1]:
## 1 — Build vocabulary from rustic_py metadata

import sys
sys.path.insert(0, '..')

from rustic_ml.autoregressive import Vocabulary

vocab = Vocabulary.from_rustic()
print(vocab.summary())

{'title': 'Size', 'field_name': 'size', 'default': 3, 'value': 3, 'min': 0, 'max': None}
Vocabulary  (23 tokens, values_width=12)
  ID  Token                             # params  Fields
--------------------------------------------------------------------------------
   0  <SOS>                                    0  
   1  <EOS>                                    0  
   2  <PAD>                                    0  
   3  <VALUES>                                 0  
   4  NOTE                                     3  note, note_on, note_off
   5  source:sine                             12  frequency, amplitude, attack, decay, … (+8)
   6  source:square                           12  frequency, amplitude, attack, decay, … (+8)
   7  source:saw                              12  frequency, amplitude, attack, decay, … (+8)
   8  source:triangle                         12  frequency, amplitude, attack, decay, … (+8)
   9  source:whitenoise                       11  amplitude, attack, decay, su

In [31]:
## 2 — Random graph generation with filters

from rustic_ml.autoregressive import random_ar_spec, spec_to_sequence
from rustic_ml.data.generation import render_mel
import IPython.display as ipd
import librosa, numpy as np

from rustic_py import GraphSpec

spec = GraphSpec.random(0.1)
print("Spec1:", len(spec.to_spec()["filters"]), "filters")
spec2 = GraphSpec.random(1.0)
print("Spec2:", len(spec2.to_spec()["filters"]), "filters")

spec = spec2.to_spec()
print("Spec filters:", [f["type"] for f in spec["filters"]])
print("Note:", spec["note"], " | Waveforms:", ",".join([f"{spec["source"]["sources"][x]["waveform"]} ({spec["source"]["sources"][x]["frequency_relation"]})" for x in range(len(spec["source"]["sources"]))]))

token_ids, values = spec_to_sequence(spec, vocab)
token_names = [vocab.id_to_token[t] for t in token_ids]
print("\nToken sequence:", token_names)
print("Values matrix shape:", values.shape)

Spec1: 1 filters
Spec2: 11 filters
Spec filters: ['HighPassFilter', 'DelayFilter', 'Compressor', 'GainFilter', 'HighPassFilter', 'Tremolo', 'Tremolo', 'LowPassFilter', 'Compressor', 'HighPassFilter', 'Tremolo']
Note: 68  | Waveforms: square (identity),square (identity)


KeyError: 'waveform'

In [ ]:
## 3 — Round-trip: spec → tokens → spec → render

from rustic_ml.autoregressive import sequence_to_spec
from rustic_py.rustic_py import render

# Encode
token_ids, values = spec_to_sequence(spec, vocab)

# Decode
spec2 = sequence_to_spec(token_ids, values, vocab)

# Verify render doesn't crash
audio = render(spec2)
print("Render OK — audio shape:", audio.shape)

# Compare key fields
print(f"\nOriginal  note={spec['note']}  waveform={spec['source']['waveform']}  "
      f"filters={[f['type'] for f in spec['filters']]}")
print(f"Decoded   note={spec2['note']}  waveform={spec2['source']['waveform']}  "
      f"filters={[f['type'] for f in spec2['filters']]}")

# Quick ADSR check
orig_adsr = {k: spec['source'][k] for k in ('attack','decay','sustain','release')}
dec_adsr  = {k: spec2['source'].get(k) for k in ('attack','decay','sustain','release')}
print("\nADSR original:", {k: f"{v:.4f}" for k, v in orig_adsr.items()})
print("ADSR decoded: ", {k: f"{v:.4f}" if v is not None else "None" for k, v in dec_adsr.items()})

In [4]:
## 4 — Dual-head AudioGraphModel

import torch
import torch.nn as nn
from torch.nn import TransformerDecoderLayer, TransformerDecoder


class AudioGraphModel(nn.Module):
    """Autoregressive graph decoder with a token head and a values head.

    Encoder: two strided Conv2d blocks collapse the mel into a single d_model
    vector that is used as transformer memory.

    Decoder: standard TransformerDecoder over teacher-forced token embeddings.
    Two output heads:
      - token_head:  Linear(d_model, vocab_size)  → next-token logits
      - values_head: Linear(d_model, values_width) → continuous param vector

    At <VALUES> positions the values_head output is used; at all other positions
    it is ignored (masked out in the loss).
    """

    def __init__(
        self,
        n_mels: int,
        vocab_size: int,
        values_width: int,
        d_model: int = 256,
        nhead: int = 8,
        num_encoder_layers: int = 4,
        num_decoder_layers: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.values_width = values_width

        # ── Mel encoder ──────────────────────────────────────────────────
        # Two Conv2d blocks halve each spatial dim → (B, 64, n_mels//4, T//4)
        # AdaptiveAvgPool squeezes time → (B, 64, n_mels//4, 1)
        self.mel_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((n_mels // 4, 1)),   # squeeze time → 1
            nn.Flatten(),                              # (B, 64 * n_mels//4)
            nn.Linear(64 * (n_mels // 4), d_model),
            nn.ReLU(),
        )

        # ── Token embedding + positional encoding ────────────────────────
        self.tok_embed = nn.Embedding(vocab_size, d_model, padding_idx=2)
        # Learnable positional encoding (max 128 tokens)
        self.pos_embed = nn.Embedding(128, d_model)

        # ── Values projection (values_width → d_model, added to token emb) ─
        self.val_proj = nn.Linear(values_width, d_model)

        # ── Transformer decoder ──────────────────────────────────────────
        dec_layer = TransformerDecoderLayer(
            d_model, nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.decoder = TransformerDecoder(dec_layer, num_decoder_layers)

        # ── Output heads ─────────────────────────────────────────────────
        self.token_head = nn.Linear(d_model, vocab_size)
        self.values_head = nn.Linear(d_model, values_width)

    def encode(self, mel: torch.Tensor) -> torch.Tensor:
        """mel: (B, 1, n_mels, T)  →  memory: (B, 1, d_model)"""
        mem = self.mel_encoder(mel)          # (B, d_model)
        return mem.unsqueeze(1)              # (B, 1, d_model)

    def forward(
        self,
        mel: torch.Tensor,
        tgt_tokens: torch.Tensor,
        tgt_values: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            mel:        (B, 1, n_mels, T)
            tgt_tokens: (B, S)      int64 — teacher-forced input token ids
            tgt_values: (B, S, W)   float32 — values at each position

        Returns:
            token_logits: (B, S, vocab_size)
            pred_values:  (B, S, values_width)
        """
        B, S = tgt_tokens.shape
        device = mel.device

        memory = self.encode(mel)       # (B, 1, d_model)

        # Token + positional + values embeddings
        positions = torch.arange(S, device=device).unsqueeze(0).expand(B, -1)
        emb = (
            self.tok_embed(tgt_tokens)          # (B, S, d_model)
            + self.pos_embed(positions)         # (B, S, d_model)
            + self.val_proj(tgt_values)         # (B, S, d_model)
        )

        # Causal mask so each position only attends to past positions
        causal_mask = nn.Transformer.generate_square_subsequent_mask(S, device=device)

        out = self.decoder(emb, memory, tgt_mask=causal_mask)  # (B, S, d_model)

        token_logits = self.token_head(out)    # (B, S, vocab_size)
        pred_values = self.values_head(out)    # (B, S, values_width)

        return token_logits, pred_values

In [5]:
## 5 — Instantiate model + sanity-check forward pass

from rustic_ml.data.generation import MEL_BINS

N_MELS = MEL_BINS  # 128

model = AudioGraphModel(
    n_mels=N_MELS,
    vocab_size=len(vocab),
    values_width=vocab.values_width,
    d_model=256,
    nhead=8,
    num_decoder_layers=4,
)
print(model)
print(f"\nvocab_size={len(vocab)}  values_width={vocab.values_width}")

# Dummy forward pass
B, S, T = 2, 10, 87
mel_dummy  = torch.randn(B, 1, N_MELS, T)
tok_dummy  = torch.randint(0, len(vocab), (B, S))
val_dummy  = torch.randn(B, S, vocab.values_width)

token_logits, pred_values = model(mel_dummy, tok_dummy, val_dummy)
print(f"\ntoken_logits: {token_logits.shape}   pred_values: {pred_values.shape}")

AudioGraphModel(
  (mel_encoder): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): AdaptiveAvgPool2d(output_size=(32, 1))
    (5): Flatten(start_dim=1, end_dim=-1)
    (6): Linear(in_features=2048, out_features=256, bias=True)
    (7): ReLU()
  )
  (tok_embed): Embedding(23, 256, padding_idx=2)
  (pos_embed): Embedding(128, 256)
  (val_proj): Linear(in_features=12, out_features=256, bias=True)
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256,

In [6]:
## 6 — Loss function + training step

def compute_loss(
    token_logits: torch.Tensor,  # (B, S, vocab_size)
    pred_values:  torch.Tensor,  # (B, S, values_width)
    tgt_tokens:   torch.Tensor,  # (B, S)   — ground-truth *output* tokens
    tgt_values:   torch.Tensor,  # (B, S, W)
    pad_id:       int,
    values_tok_id: int,
    lambda_values: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Combined cross-entropy (tokens) + masked MSE (values).

    The values loss is only computed at positions where tgt_tokens == <VALUES>.

    Returns:
        total_loss, token_loss, values_loss
    """
    B, S, V = token_logits.shape

    # Token loss: reshape to (B*S, vocab_size) vs (B*S,)
    tok_loss = torch.nn.functional.cross_entropy(
        token_logits.reshape(-1, V),
        tgt_tokens.reshape(-1),
        ignore_index=pad_id,
    )

    # Values loss: only at <VALUES> positions
    values_mask = (tgt_tokens == values_tok_id)  # (B, S) bool
    if values_mask.any():
        # MSE between predicted and target values, averaged over active positions
        masked_pred = pred_values[values_mask]    # (N_values, W)
        masked_tgt  = tgt_values[values_mask]     # (N_values, W)
        val_loss = torch.nn.functional.mse_loss(masked_pred, masked_tgt)
    else:
        val_loss = torch.zeros(1, device=token_logits.device).squeeze()

    total = tok_loss + lambda_values * val_loss
    return total, tok_loss, val_loss


def train_step(model, batch, optimizer, lambda_values=1.0):
    model.train()
    optimizer.zero_grad()

    mel        = batch["mel"].unsqueeze(1)          # (B, 1, MEL_BINS, T)
    token_ids  = batch["token_ids"]                 # (B, S)
    values_mat = batch["values"]                    # (B, S, W)

    # Teacher forcing: input = all but last, target = all but first
    tgt_input  = token_ids[:, :-1]
    tgt_target = token_ids[:, 1:]
    val_input  = values_mat[:, :-1]
    val_target = values_mat[:, 1:]

    token_logits, pred_values = model(mel, tgt_input, val_input)

    loss, tok_loss, val_loss = compute_loss(
        token_logits, pred_values,
        tgt_target, val_target,
        pad_id=vocab.pad,
        values_tok_id=vocab.values_tok,
        lambda_values=lambda_values,
    )
    loss.backward()
    optimizer.step()
    return float(loss), float(tok_loss), float(val_loss)

In [ ]:
## 7 — Training loop with ARDataset

from torch.utils.data import DataLoader
from rustic_ml.autoregressive import ARDataset, ar_collate_fn

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

dataset = ARDataset(n_samples=200, vocab=vocab, max_filters=3)
dataloader = DataLoader(dataset, batch_size=8, collate_fn=ar_collate_fn, num_workers=0)

for epoch in range(3):
    for batch in dataloader:
        loss, tok_loss, val_loss = train_step(model, batch, optimizer)
    print(f"Epoch {epoch}  total={loss:.4f}  token={tok_loss:.4f}  values={val_loss:.4f}")

In [ ]:
## 8 — Autoregressive inference (greedy decoding)

@torch.no_grad()
def generate_graph(model, mel_np, vocab, max_len=64, device="cpu"):
    """Greedy autoregressive decoding from a mel spectrogram.

    Returns:
        spec_dict: decoded GraphSpec compatible with rustic_py.render()
        token_names: list of token name strings
    """
    import numpy as np
    model.eval()

    mel = torch.from_numpy(mel_np).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,M,T)

    token_ids = [vocab.sos]
    values_rows = [np.zeros(vocab.values_width, dtype=np.float32)]

    for _ in range(max_len):
        tok_t  = torch.tensor([token_ids], dtype=torch.int64, device=device)       # (1, S)
        val_t  = torch.tensor([values_rows], dtype=torch.float32, device=device)   # (1, S, W)

        tok_logits, pred_vals = model(mel, tok_t, val_t)  # (1, S, V), (1, S, W)

        # Pick next token from the last position
        next_tok = int(tok_logits[0, -1].argmax())
        next_val = pred_vals[0, -1].cpu().numpy()

        token_ids.append(next_tok)
        values_rows.append(next_val if next_tok == vocab.values_tok else np.zeros(vocab.values_width))

        if next_tok == vocab.eos:
            break

    values_matrix = np.stack(values_rows, axis=0)
    spec = sequence_to_spec(token_ids, values_matrix, vocab)
    token_names = [vocab.id_to_token.get(t, "?") for t in token_ids]
    return spec, token_names


# Demo inference on a freshly rendered spec
test_spec = random_ar_spec(vocab, max_filters=2)
test_mel  = render_mel(test_spec)

pred_spec, pred_tokens = generate_graph(model, test_mel, vocab)
print("Predicted tokens:", pred_tokens)
print("Predicted spec filters:", [f["type"] for f in pred_spec["filters"]])